In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive')
os.chdir('/content/gdrive/My Drive/CS595A/Project/YOLOv3-Custom-Object-Detection')
!ls

Mounted at /content/gdrive
 classes.txt	       YOLOv3_Custom_Object_Detection.ipynb
 Object_Detection.py   yolov3_testing.cfg
 test_images	       yolov3_training_last.weights
'Use Case Examples'


In [ ]:
! ls

 classes.txt	       YOLOv3_Custom_Object_Detection.ipynb
 Object_Detection.py   yolov3_testing.cfg
 test_images	       yolov3_training_last.weights
'Use Case Examples'


In [ ]:
!cd test_images

In [ ]:
! ls

 classes.txt	       YOLOv3_Custom_Object_Detection.ipynb
 Object_Detection.py   yolov3_testing.cfg
 test_images	       yolov3_training_last.weights
'Use Case Examples'


In [ ]:
! python Object_Detection.py

^C


In [ ]:
import cv2
import numpy as np
import os

net = cv2.dnn.readNet('yolov3_training_last.weights', 'yolov3_testing.cfg')

classes = []
with open("classes.txt", "r") as f:
    classes = f.read().splitlines()

font = cv2.FONT_HERSHEY_SIMPLEX
colors = {
    "Actor": (0,255,0),
    "Extends": (0,0,255),
    "Includes": (255,0,0),
    "Use Case": (255,0,255)
}
path = "test_images"
save_path = 'Use_case_examples'
for imagePath in os.listdir(path):
    imgLoc = os.path.join(path, imagePath)
    img = cv2.imread(imgLoc)
    height, width, _ = img.shape

    blob = cv2.dnn.blobFromImage(img, 1/255, (416, 416), (0,0,0), swapRB=True, crop=False)
    net.setInput(blob)
    output_layers_names = net.getUnconnectedOutLayersNames()
    layerOutputs = net.forward(output_layers_names)

    boxes = []
    confidences = []
    class_ids = []

    for output in layerOutputs:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > 0.2:
                center_x = int(detection[0]*width)
                center_y = int(detection[1]*height)
                w = int(detection[2]*width)
                h = int(detection[3]*height)
                x = int(center_x - w/2)
                y = int(center_y - h/2)

                boxes.append([x, y, w, h])
                confidences.append((float(confidence)))
                class_ids.append(class_id)

    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.2, 0.4)

    if len(indexes)>0:
        for i in indexes.flatten():
            x, y, w, h = boxes[i]
            label = str(classes[class_ids[i]])
            confidence = str(round(confidences[i],2))
            color = colors[label]
            cv2.rectangle(img, (x,y), (x+w, y+h), color, 2)
            cv2.putText(img, label + " " + confidence, (x+5, y+10), font, 0.5, (0,0,255), 1)
    save_filename = 'analyzed_' + imagePath
    save_path = os.path.join(save_path, save_filename)
    cv2.imwrite(save_path, img)
    key = cv2.waitKey(1)
    if key==27:
        break

##cap.release()
cv2.destroyAllWindows()